# Ejecutar el monitor de DeltaPay

Ejecutá la celda con ▶. Descarga el script y los CSV de GitHub, simula agosto a octubre de 2026 y muestra los resultados con y sin cupo. En **Archivos → `tapi_monitor` → `resultados_cupo`** están los siete CSV.


In [ ]:
import csv
from pathlib import Path
import subprocess
import sys
from urllib.request import urlretrieve

url_base = "https://raw.githubusercontent.com/DaianaSep/tapi_Challenge/main"
carpeta = Path("tapi_monitor")
carpeta.mkdir(exist_ok=True)
for nombre in ("02_monitor_ratio.py", "B_agenda_consultas.csv", "B_agenda_pagos.csv"):
    urlretrieve(f"{url_base}/{nombre}", carpeta / nombre)

salida = carpeta / "resultados_cupo"
subprocess.run([
    sys.executable, str(carpeta / "02_monitor_ratio.py"),
    "--datos", str(carpeta), "--salida", str(salida),
    "--desde", "2026-08-03", "--hasta", "2026-10-25",
    "--aprobar-cupo", "2026-08-04=5",
    "--aprobar-cupo", "2026-09-03=70",
    "--aprobar-cupo", "2026-10-06=140",
], check=True)

print(f"\nArchivos generados en {salida}:")
for archivo in sorted(salida.glob("*.csv")):
    print(archivo.name)

print("\nSemana     | Pagos | Ratio sin cupo | Ratio con cupo")
with (salida / "resumen_semanal.csv").open(encoding="utf-8", newline="") as archivo:
    for fila in csv.DictReader(archivo):
        print(f"{fila['semana']} | {fila['pagos']:>5} | "
              f"{float(fila['ratio_monitor']):>14.2f} | {float(fila['ratio_cupo']):>14.2f}")

print("\nFecha      | Consultas | Pagos | Margen | Cupo del día")
with (salida / "controles_diarios_cupo.csv").open(encoding="utf-8", newline="") as archivo:
    for fila in csv.DictReader(archivo):
        if "2026-09-03" <= fila["fecha"] <= "2026-09-06":
            print(f"{fila['fecha']} | {fila['consultas']:>9} | {fila['pagos']:>5} | "
                  f"{fila['margen_consultas']:>6} | {fila['cupo_busquedas_dia']:>12}")
